# Hosaki函数

**类别：** 仿真

来源：[https://www.hexaly.com/templates/hosaki-function](https://www.hexaly.com/templates/hosaki-function)


## 问题

**Hosaki函数**由下式定义：

  f
  (
  x
  )
  =
  (
  1
  −
  8
  
    x
    1
  
  +
  7
  
    x
    1
    2
  
  −
  
    7
    3
  
  
    x
    1
    3
  
  +
  
    1
    4
  
  
    x
    1
    4
  
  )
  
    x
    2
    2
  
  
    e
    
      −
      
        x
        2
      
    
  

这是一个盒约束问题。变量 x1 和 x2 的定义域分别为 [0, 5] 和 [0, 6]。问题的目标是找到该函数的最小值。更多细节，请参阅 [hosaki_function.html](http://jakobbossek.github.io/smoof/reference/makeHosakiFunction.html)。

	

### 学到的建模原则

- 创建一个 [external function](https://www.hexaly.com/docs/last/mathematicaloperators/externalfunctions.html)
- 在 external function 上启用 [surrogate modeling](https://www.hexaly.com/docs/last/mathematicaloperators/externalfunctions.html#surrogate-modeling)
- 为该函数设置 evaluation limit

Hosaki函数问题可以在不使用 surrogate modeling 功能的情况下求解（参见 [Branin function](https://www.hexaly.com/example/branin-function)）。事实上，当目标函数计算代价高昂时，该功能非常有用。本示例的目的只是在一个简单且计算代价低廉的问题上演示 surrogate modeling 的使用。


## 模型

Hosaki函数问题的Hexaly模型使用两个 float decision variables：x1 和 x2。这些变量的定义域分别为 [0, 5] 和 [0, 6]。

该问题没有任何约束条件，只有一个需要最小化的目标函数。目标函数由 external function 定义。它通过 HxExternalArgumentValues 接收参数值（x1 和 x2），并返回该函数在该点的值。为了计算 external function 返回的值，需要创建一个 O_Call 表达式，然后对其进行最小化。

为了使用 [surrogate modeling feature](https://www.hexaly.com/docs/last/mathematicaloperators/externalfunctions.html#external-functions-surrogate-modeling)，需要在函数的 HxExternalContext 上调用 enableSurrogateModeling 方法。此方法返回 HxSurrogateParameters，可以在其上设置函数的最大调用次数。由于该函数在实际应用中通常计算代价高昂，因此将搜索限制在合理的时间内非常有用。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys
import math

#
# External function
#
def hosaki_function(argument_values):
    x1 = argument_values[0]
    x2 = argument_values[1]
    return ((1 - 8 * x1 + 7 * pow(x1, 2) - 7 * pow(x1, 3) / 3 + pow(x1, 4) / 4) 
            * pow(x2, 2) * math.exp(-x2))


def main(evaluation_limit, output_file):
    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        #
        # Declare the optimization model
        #
        model = optimizer.model

        # Numerical decisions
        x1 = model.float(0, 5)
        x2 = model.float(0, 6)

        # Create and call the function
        f = model.create_double_external_function(hosaki_function)
        func_call = model.call(f, x1, x2)

        # Enable surrogate modeling
        surrogate_params = f.external_context.enable_surrogate_modeling()

        # Minimize function call
        model.minimize(func_call)
        model.close()

        # Parameterize the optimizer
        surrogate_params.evaluation_limit = evaluation_limit

        optimizer.solve()

        # Write the solution in a file
        if output_file is not None:
            with open(output_file, 'w') as f:
                f.write("obj=%f\n" % func_call.value)
                f.write("x1=%f\n" % x1.value)
                f.write("x2=%f\n" % x2.value)


if __name__ == '__main__':
    output_file = sys.argv[1] if len(sys.argv) > 1 else None
    evaluation_limit = int(sys.argv[2]) if len(sys.argv) > 2 else 30

    main(evaluation_limit, output_file)
